# D2.2 · When the actor is an agent

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.1 · Agent-assisted reconstruction](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**.

| | |
|---|---|
| Tools used | Keycloak, OpenSearch |

## What this lesson is

**What it covers.** Attribute an incident through the A2 `act` chain.

**Why a security engineer needs it.** "Which user" is now the wrong first question. The control it builds is: attribute to agent, authority, delegation chain and prompt.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The internal actor was autonomous. Was it instructed, was it compromised, or did it simply do what it was allowed to do? None of your existing playbooks have a branch for that question, and the answer changes everything downstream.

> **At CyberTravels.** The internal actor was the Workflow Agent. Was it instructed, injected, or simply permitted? CyberTravels' existing playbook has no branch for that question, and every step of it assumes a person.

## 2 · The framework

```
   the internal actor was autonomous. which branch?

   instructed      someone told it to        -> who, and through what channel
   injected        content told it to        -> which corpus, written by whom
   permitted       it was allowed to         -> a control gap, not an intrusion

   no existing playbook has this branch, and it changes everything after it
```

Three responder instincts are correct for human incidents and misfire when the
actor is an agent.

1. **Disable the account.** For a human this stops them. For an agent holding an
   already-issued bearer token, it may not — the token remains valid until it
   expires.
2. **Interview the user.** They were asleep. They authorised a task; a model
   chose the actions. They cannot tell you what happened.
3. **Assume one actor.** There were three, in a chain, and only the last one
   touched the resource.

The correct first action is to **revoke the agent identity**, which is only
possible if A2 was done. This lesson is where the identity track's value becomes
operational rather than architectural.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the runbook you have</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the runbook this incident needs</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">1. disable the user account</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">1. identify the &lt;b&gt;acting&lt;/b&gt; identity from the act chain (A2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">2. interview the user</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">2. revoke that identity — no approval needed for a non-human (A3.6)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">3. review the user&#x27;s recent activity</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">3. scope by walking the delegation chain, not the host list (D2.3)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">4. preserve the run trace before anything restarts (D2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">5. only then consider the human&#x27;s account, and say why</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Every step on the left is correct for a human actor and wrong here. The human authorised a task; the actions were chosen by a model.</div>

## 3 · The procedure, as a skill

Disabling the human's account leaves both agents acting on tokens already issued. The skill enumerates the live sessions, simulates the reflex containment step, and separates the task the user authorised from the actions taken under it.

In [ ]:
# skills/response/agent-actor-containment/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-actor-containment
description: >-
  Establish what disabling a human's account does not stop when agents hold
  issued tokens, and separate the task a user authorised from the actions taken
  under it. Use during an incident where the actor is an agent and the account
  is a person's.
allowed-tools: Read, Grep, Glob
---

# Disabling her account stops her, not them

The reflex containment step is to disable the account the logs name. When the
actor is an agent holding an already-issued token, that step changes nothing:
the token is valid until it expires, and the agent keeps acting. Containment has
to name the credential and the workload, not the person.

## When to use this

Any incident where an agent acted on a user's behalf, and before any containment
decision that starts by disabling an account.

## Procedure

**1 — Enumerate live sessions and issued tokens per actor.** Human sessions,
agent sessions, and the tokens each holds with their expiry. This list is the
containment surface and it is usually longer than expected.

**2 — Simulate disabling the human's account.** Record what stops and what does
not. Already-issued tokens continuing to work is the finding, and it needs to be
stated before the containment call is made.

**3 — Interview to separate authorisation from action.** The user authorised a
*task*. Which of the actions taken under it did they know about, ask for, or
see? The answer is usually "the first one", and it changes the incident's
character entirely.

**4 — Draw the actor chain.** User, orchestrator, worker agent, downstream. The
logs show one actor; the chain shows three. Name each and what each can still
do.

**5 — Choose levers by what they actually stop.** Account disable, token
revocation, workload termination, downstream block. Record the effect and the
collateral of each, and pick from that table rather than from habit.

## Output contract

```json
{
  "sessions": [{"actor": "str", "kind": "human|agent", "tokens": [{"id": "str", "expires_in_s": 0}]}],
  "disable_human": {"stops": ["str"], "does_not_stop": ["str"]},
  "interview": {"authorised": "str", "aware_of": ["str"], "unaware_of": ["str"]},
  "chain": ["str"],
  "levers": [{"lever": "str", "stops": ["str"], "collateral": ["str"]}]
}
```

## Failure modes

- **Starting with the account.** It is the one lever that does not touch the
  actor.
- **Recording the user as having authorised the actions.** They authorised a
  task.
- **Containing the worker and not the orchestrator.** It will start another.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/response/agent-actor-containment/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/response/agent-actor-containment/scripts/agent_actor_containment.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Establish what a human's account being disabled does not stop, and separate a task the user authorised from the actions taken under it.

This is the executable half of the `agent-actor-containment` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
from dataclasses import dataclass, field

@dataclass
class Session:
    actor: str; token_issued: float; token_ttl: float; account_enabled: bool = True
    identity_revoked: bool = False
    def can_act(self, at):
        if self.identity_revoked: return False, "identity revoked"
        if at - self.token_issued > self.token_ttl: return False, "token expired"
        if not self.account_enabled:
            return True, "account disabled, but the issued token is still valid"
        return True, "active"

now = time.time()
SESSIONS = {
 "dana@corp (human)":  Session("dana@corp", now-60, 3600),
 "patch-agent":        Session("patch-agent", now-60, 3600),
 "deploy-agent":       Session("deploy-agent", now-60, 3600),
}
print("INSTINCT 1 — disable dana@corp's account")
for s in SESSIONS.values(): s.account_enabled = (s.actor != "dana@corp")
for name, s in SESSIONS.items():
    ok, why = s.can_act(now)
    print(f"   {name:22s} can act: {str(ok):5s}  {why}")
print("   → the agents were never using her account interactively; they hold")
print("     their own issued tokens, and one of them is acting AS her.")

print("\nINSTINCT 2 — interview the user")
INTERVIEW = {
 "did you read the AWS credentials?":       "No. I opened a ticket and went to lunch.",
 "what did you ask the agent to do?":       "Fix the finding in billing.py.",
 "did you approve the external POST?":      "I didn't know it made external calls.",
}
for q, a in INTERVIEW.items():
    print(f"   Q: {q}\n   A: {a}")
print("   → she authorised a TASK. The actions were chosen by a model. She is")
print("     not withholding information; she does not have it.")

print("\nINSTINCT 3 — assume one actor")
CHAIN = ["dana@corp", "orchestrator", "patch-agent"]
print(f"   actual chain: {' → '.join(CHAIN)}")
print(f"   actors involved: {len(CHAIN)}; actors in the logs: 1")

class Registry:
    def __init__(self):
        self.revoked = set()
    def revoke(self, actor):
        self.revoked.add(actor); return actor
    def valid(self, session):
        return session.actor not in self.revoked

reg = Registry()
for s in SESSIONS.values(): s.account_enabled = True   # undo instinct 1

print("correct first action — revoke patch-agent's identity:")
reg.revoke("patch-agent")
SESSIONS["patch-agent"].identity_revoked = True
for name, s in SESSIONS.items():
    ok, why = s.can_act(now)
    print(f"   {name:22s} can act: {str(ok):5s}  {why}")
print("\n   dana keeps working. deploy-agent keeps working. The actor stopped.")

print("\nTIME TO EFFECT, measured:")
LEVERS = {"disable the human's account": (5,  "agent unaffected"),
          "kill the agent process":      (2,  "supervisor restarts it; token still valid"),
          "revoke the agent identity":   (12, "agent cannot act, even after restart"),
          "rotate the shared credential":(420,"works, and breaks every other consumer")}
for lever, (secs, note) in LEVERS.items():
    print(f"   {lever:32s}{secs:>5}s  {note}")
assert not SESSIONS["patch-agent"].can_act(now)[0]
assert SESSIONS["deploy-agent"].can_act(now)[0]

## What you just proved

Disabling the human's account leaves both agents able to act on already-issued tokens. The interview establishes the user authorised a task, not the actions. The chain shows three actors where the logs show one. Revoking `patch-agent`'s identity stops it in 12 seconds while dana and `deploy-agent` continue working.

## Your turn

Write your agentic incident runbook's first three steps. If step one is "disable the user account", rewrite it — and check whether you can currently revoke a single agent identity at all.

---

**Next → [D2.3 · Scoping an agentic incident](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*